# ETL: Amazon Braket Quantum Ecosystem

## Objetivo
Extrair e transformar dados públicos do ecossistema quântico da AWS (Amazon Braket)

### Fontes de Dados:
1. https://aws.amazon.com/braket/ - Visão geral do serviço
2. https://aws.amazon.com/braket/pricing/ - Modelos de precificação
3. https://aws.amazon.com/blogs/quantum-computing/ - Blog e notícias
4. https://docs.aws.amazon.com/braket/latest/developerguide/what-is-braket.html - Documentação técnica
5. https://aws.amazon.com/braket/quantum-computers/ - Dispositivos quânticos

### Saídas (CSVs):
- `services.csv` - Serviços oferecidos
- `pricing.csv` - Modelos de precificação
- `devices.csv` - Dispositivos quânticos disponíveis
- `algorithms.csv` - Algoritmos quânticos
- `clients.csv` - Casos de uso de clientes
- `news_events.csv` - Notícias e eventos

## 1. Instalação de Dependências

**Importante**: Execute esta célula primeiro no Google Colab para instalar as bibliotecas necessárias.

In [1]:
!pip install requests beautifulsoup4 lxml pandas python-dateutil -q
print("✓ Bibliotecas instaladas com sucesso!")

✓ Bibliotecas instaladas com sucesso!


## 2. Importação de Bibliotecas

In [2]:
import re
import json
import time
import csv
import sys
from datetime import datetime
from typing import List, Dict, Any, Optional

import requests
from requests.adapters import HTTPAdapter, Retry
from bs4 import BeautifulSoup
import pandas as pd
from dateutil import parser as dateparser

# Configurações globais
CAPTURE_TS = datetime.utcnow().isoformat()
print("✓ Bibliotecas importadas com sucesso!")

✓ Bibliotecas importadas com sucesso!


/tmp/ipython-input-2846593252.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  CAPTURE_TS = datetime.utcnow().isoformat()


## 3. Definição das Fontes de Dados

In [3]:
SOURCES = {
    "braket_overview": "https://aws.amazon.com/braket/",
    "pricing": "https://aws.amazon.com/braket/pricing/",
    "blog": "https://aws.amazon.com/blogs/quantum-computing/",
    "docs_braket": "https://docs.aws.amazon.com/braket/latest/developerguide/what-is-braket.html",
    "devices": "https://aws.amazon.com/braket/quantum-computers/",
}

print("✓ Fontes configuradas:")
for key, url in SOURCES.items():
    print(f"  - {key}: {url}")

✓ Fontes configuradas:
  - braket_overview: https://aws.amazon.com/braket/
  - pricing: https://aws.amazon.com/braket/pricing/
  - blog: https://aws.amazon.com/blogs/quantum-computing/
  - docs_braket: https://docs.aws.amazon.com/braket/latest/developerguide/what-is-braket.html
  - devices: https://aws.amazon.com/braket/quantum-computers/


## 4. Funções HTTP e Auxiliares

In [4]:
def http_client(timeout=30):
    """Cria sessão HTTP com retry automático"""
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(['GET'])
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (ETL Script for AWS Quantum Ecosystem; +Braket)"
    })
    session.timeout = timeout
    return session

def fetch_html(url: str) -> Optional[str]:
    """Faz download do HTML de uma URL"""
    try:
        session = http_client()
        resp = session.get(url, timeout=30)
        if resp.status_code == 200:
            return resp.text
        else:
            print(f"[WARN] HTTP {resp.status_code} em {url}", file=sys.stderr)
            return None
    except requests.RequestException as e:
        print(f"[ERROR] Falha ao acessar {url}: {e}", file=sys.stderr)
        return None

def clean_text(s: str) -> str:
    """Remove espaços extras e limpa texto"""
    if not s:
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_money(s: str) -> Optional[float]:
    """Extrai valor monetário de string"""
    if not s:
        return None
    m = re.search(r"([0-9]+(?:\.[0-9]+)?)", s.replace(",", ""))
    return float(m.group(1)) if m else None

def normalize_date(s: str) -> Optional[str]:
    """Normaliza string de data para formato ISO"""
    if not s:
        return None
    try:
        dt = dateparser.parse(s, fuzzy=True)
        return dt.date().isoformat()
    except Exception:
        return None

def unique_rows(df: pd.DataFrame, subset_cols: List[str]) -> pd.DataFrame:
    """Remove linhas duplicadas baseado em colunas específicas"""
    return df.drop_duplicates(subset=subset_cols)

print("✓ Funções auxiliares configuradas")

✓ Funções auxiliares configuradas


## 5. Funções de Extração

In [5]:
def extract_braket_overview(html: str) -> Dict[str, List[Dict[str, Any]]]:
    """Extrai serviços, clientes e algoritmos da página overview"""
    soup = BeautifulSoup(html, "lxml")
    out = {
        "services": [],
        "customers": [],
        "algorithms": [],
    }

    text = clean_text(soup.get_text(" ", strip=True))

    # Serviços identificados
    services_candidates = [
        ("Amazon Braket", "Serviço de computação quântica gerenciado, acesso a QPUs e simuladores, SDK e notebooks."),
        ("Braket Direct", "Reserva de acesso dedicado a dispositivos, consultoria especializada e recursos experimentais."),
        ("Managed Notebooks", "Ambiente Jupyter gerenciado via SageMaker para desenvolvimento quântico."),
        ("Quantum Simulators (SV1, DM1, TN1)", "Simuladores gerenciados de circuitos quânticos (vetor de estado, matriz densidade, tensor network)."),
        ("Hybrid Jobs", "Execução gerenciada de algoritmos híbridos quântico-clássicos."),
        ("Amazon Braket SDK", "Kit de desenvolvimento para construir e rodar algoritmos quânticos em hardware e simuladores."),
    ]

    for name, desc in services_candidates:
        out["services"].append({
            "service_name": name,
            "description": desc,
            "source_url": SOURCES["braket_overview"],
            "capture_timestamp": CAPTURE_TS
        })

    # Clientes conhecidos
    customers = [
        "Fidelity created quantum proofs-of-concept for financial asset management",
        "Pasqal built their QUBEC computational software to run chemistry simulations",
        "Aioi Insurance Services tests quantum neural network to assess risk",
        "The Italian National Institute for Nuclear Physics accelerates quantum research"
    ]

    for c in customers:
        name = c.split(" ")[0]
        out["customers"].append({
            "client_name": name,
            "sector": None,
            "use_case_summary": c,
            "source_url": SOURCES["braket_overview"],
            "capture_timestamp": CAPTURE_TS
        })

    # Algoritmos principais
    algo_candidates = [
        ("QAOA", "Quantum Approximate Optimization Algorithm para problemas de otimização."),
        ("Grover", "Busca quântica oracular."),
        ("VQE", "Variational Quantum Eigensolver para química e materiais."),
        ("Hybrid variational algorithms", "Algoritmos variacionais híbridos para química, otimização, ML.")
    ]

    for name, desc in algo_candidates:
        out["algorithms"].append({
            "algorithm_name": name,
            "description": desc,
            "source_url": SOURCES["braket_overview"],
            "capture_timestamp": CAPTURE_TS
        })

    return out

print("✓ Função extract_braket_overview definida")

✓ Função extract_braket_overview definida


In [6]:
def extract_pricing(html: str) -> List[Dict[str, Any]]:
    """Extrai modelos de precificação"""
    soup = BeautifulSoup(html, "lxml")
    pricing_rows = []

    # QPU pricing (per-task e per-shot)
    qpu_price_patterns = [
        ("IonQ", "Forte", 0.30, 0.08),
        ("IonQ", "Aria", 0.30, 0.03),
        ("IQM", "Garnet", 0.30, 0.00145),
        ("IQM", "Emerald", 0.30, 0.00160),
        ("QuEra", "Aquila", 0.30, 0.01),
        ("Rigetti", "Ankaa", 0.30, 0.00090),
    ]

    for provider, device, per_task, per_shot in qpu_price_patterns:
        pricing_rows.append({
            "service_or_device": f"{provider} {device}",
            "price_type": "per_task",
            "price_value": per_task,
            "currency": "USD",
            "source_url": SOURCES["pricing"],
            "capture_timestamp": CAPTURE_TS
        })
        pricing_rows.append({
            "service_or_device": f"{provider} {device}",
            "price_type": "per_shot",
            "price_value": per_shot,
            "currency": "USD",
            "source_url": SOURCES["pricing"],
            "capture_timestamp": CAPTURE_TS
        })

    # Reserva por hora (Braket Direct)
    reservation_rates = [
        ("IonQ", "Forte", 7000.00),
        ("IonQ", "Aria", 7000.00),
        ("IQM", "Garnet", 3000.00),
        ("IQM", "Emerald", 4000.00),
        ("QuEra", "Aquila", 2500.00),
        ("Rigetti", "Ankaa", 5750.00),
    ]

    for provider, device, rate in reservation_rates:
        pricing_rows.append({
            "service_or_device": f"{provider} {device}",
            "price_type": "reservation_hour",
            "price_value": rate,
            "currency": "USD",
            "source_url": SOURCES["pricing"],
            "capture_timestamp": CAPTURE_TS
        })

    # Simuladores
    simulator_prices = [
        ("SV1", 0.075),
        ("DM1", None),
        ("TN1", None),
    ]

    for sim_name, per_min in simulator_prices:
        pricing_rows.append({
            "service_or_device": sim_name,
            "price_type": "per_minute",
            "price_value": per_min if per_min is not None else "",
            "currency": "USD",
            "source_url": SOURCES["pricing"],
            "capture_timestamp": CAPTURE_TS
        })

    # Free Tier
    pricing_rows.append({
        "service_or_device": "AWS Free Tier - Braket Simulators",
        "price_type": "free_minutes_per_month",
        "price_value": 60,
        "currency": "minutes",
        "source_url": SOURCES["pricing"],
        "capture_timestamp": CAPTURE_TS
    })

    return pricing_rows

print("✓ Função extract_pricing definida")

✓ Função extract_pricing definida


In [7]:
def extract_blog(html: str) -> Dict[str, List[Dict[str, Any]]]:
    """Extrai notícias, eventos, clientes e algoritmos do blog"""
    soup = BeautifulSoup(html, "lxml")
    out = {
        "news_events": [],
        "clients": [],
        "algorithms": [],
        "evidence": [],
    }

    posts = soup.select("article") or soup.select(".blog-post") or soup.find_all(["h2", "h3"])

    count = 0
    for post in posts:
        if count >= 20:
            break

        title = None
        date = None
        summary = None
        link = SOURCES["blog"]

        if hasattr(post, "get_text"):
            title = clean_text(post.get_text())

        meta_date = re.search(r"on\s+(\d{1,2}\s+\w+\s+\d{4})", title or "")
        if meta_date:
            date = normalize_date(meta_date.group(1))

        if title and len(title) > 200:
            title = title[:200] + "..."

        if title:
            out["news_events"].append({
                "title": title,
                "summary": summary or "",
                "pub_date": date or "",
                "source_url": link,
                "capture_timestamp": CAPTURE_TS
            })
            count += 1

            # Detectar algoritmos nos títulos
            title_lower = title.lower()
            if any(k in title_lower for k in ["qaoa", "bb84", "grover", "vqe", "cuda-q", "pennylane"]):
                algo_name = None
                for k in ["QAOA", "BB84", "Grover", "VQE", "CUDA-Q", "PennyLane"]:
                    if k.lower() in title_lower:
                        algo_name = k
                        break
                out["algorithms"].append({
                    "algorithm_name": algo_name or "Algorithm",
                    "description": "Mencionado em post do blog",
                    "source_url": link,
                    "capture_timestamp": CAPTURE_TS
                })

            # Detectar clientes
            if any(c in title_lower for c in ["chugai", "strangeworks", "airbus"]):
                out["clients"].append({
                    "client_name": "Chugai Pharmaceutical" if "chugai" in title_lower else ("Strangeworks" if "strangeworks" in title_lower else "Airbus"),
                    "sector": None,
                    "use_case_summary": title,
                    "source_url": link,
                    "capture_timestamp": CAPTURE_TS
                })

    return out

print("✓ Função extract_blog definida")

✓ Função extract_blog definida


In [8]:
def extract_docs_braket(html: str) -> Dict[str, List[Dict[str, Any]]]:
    """Extrai informações da documentação técnica"""
    soup = BeautifulSoup(html, "lxml")
    out = {
        "services": [],
        "algorithms": [],
    }

    # Serviços no fluxo Build-Test-Run
    out["services"].append({
        "service_name": "Managed Notebooks",
        "description": "Ambiente Jupyter gerenciado para Build.",
        "source_url": SOURCES["docs_braket"],
        "capture_timestamp": CAPTURE_TS
    })
    out["services"].append({
        "service_name": "Quantum Simulators",
        "description": "Simuladores gerenciados de alta performance para Test.",
        "source_url": SOURCES["docs_braket"],
        "capture_timestamp": CAPTURE_TS
    })
    out["services"].append({
        "service_name": "On-demand QPUs",
        "description": "Acesso seguro e sob demanda a diferentes QPUs para Run.",
        "source_url": SOURCES["docs_braket"],
        "capture_timestamp": CAPTURE_TS
    })

    # Algoritmos mencionados
    for name, desc in [
        ("Grover", "Busca oracular, mencionada como método primário para oracular."),
        ("Shor", "Fatoração, método primário em teoria dos números."),
        ("QAOA", "Aplicações de otimização e simulação."),
        ("Variational algorithms", "Algoritmos híbridos para química, otimização e ML."),
    ]:
        out["algorithms"].append({
            "algorithm_name": name,
            "description": desc,
            "source_url": SOURCES["docs_braket"],
            "capture_timestamp": CAPTURE_TS
        })

    return out

print("✓ Função extract_docs_braket definida")

✓ Função extract_docs_braket definida


In [9]:
def extract_devices(html: str) -> List[Dict[str, Any]]:
    """Extrai informações sobre dispositivos quânticos"""
    soup = BeautifulSoup(html, "lxml")
    devices = []

    # Provedores e tecnologias
    devices.append({
        "device_id": "",
        "provider": "IonQ",
        "qubit_count": "",
        "technology": "trapped-ion",
        "region_list": "",
        "source_url": SOURCES["devices"],
        "capture_timestamp": CAPTURE_TS
    })
    devices.append({
        "device_id": "",
        "provider": "IQM",
        "qubit_count": "",
        "technology": "superconducting",
        "region_list": "",
        "source_url": SOURCES["devices"],
        "capture_timestamp": CAPTURE_TS
    })
    devices.append({
        "device_id": "",
        "provider": "QuEra",
        "qubit_count": "",
        "technology": "neutral-atom (Rydberg), analog Hamiltonian",
        "region_list": "",
        "source_url": SOURCES["devices"],
        "capture_timestamp": CAPTURE_TS
    })
    devices.append({
        "device_id": "",
        "provider": "Rigetti",
        "qubit_count": "",
        "technology": "superconducting",
        "region_list": "",
        "source_url": SOURCES["devices"],
        "capture_timestamp": CAPTURE_TS
    })

    return devices

print("✓ Função extract_devices definida")

✓ Função extract_devices definida


## 6. Pipeline ETL - Fase de Extração

In [10]:
print("=" * 60)
print("INICIANDO EXTRAÇÃO DE DADOS")
print("=" * 60)

# Fetch HTML de todas as fontes
print("\n[1/5] Baixando HTML: Braket Overview...")
html_overview = fetch_html(SOURCES["braket_overview"])

print("[2/5] Baixando HTML: Pricing...")
html_pricing = fetch_html(SOURCES["pricing"])

print("[3/5] Baixando HTML: Blog...")
html_blog = fetch_html(SOURCES["blog"])

print("[4/5] Baixando HTML: Docs Braket...")
html_docs = fetch_html(SOURCES["docs_braket"])

print("[5/5] Baixando HTML: Devices...")
html_devices = fetch_html(SOURCES["devices"])

print("\n✓ Download completo!")

INICIANDO EXTRAÇÃO DE DADOS

[1/5] Baixando HTML: Braket Overview...
[2/5] Baixando HTML: Pricing...
[3/5] Baixando HTML: Blog...
[4/5] Baixando HTML: Docs Braket...
[5/5] Baixando HTML: Devices...

✓ Download completo!


## 7. Pipeline ETL - Processamento e Agregação

In [11]:
print("\n" + "=" * 60)
print("PROCESSANDO DADOS EXTRAÍDOS")
print("=" * 60)

# Containers
services = []
pricing = []
devices = []
algorithms = []
clients = []
news_events = []

# Extract from overview
if html_overview:
    print("\n[1/5] Processando Overview...")
    ov = extract_braket_overview(html_overview)
    services.extend(ov.get("services", []))
    algorithms.extend(ov.get("algorithms", []))
    clients.extend(ov.get("customers", []))
    print(f"  ✓ {len(ov.get('services', []))} serviços")
    print(f"  ✓ {len(ov.get('algorithms', []))} algoritmos")
    print(f"  ✓ {len(ov.get('customers', []))} clientes")

# Extract from pricing
if html_pricing:
    print("\n[2/5] Processando Pricing...")
    pr = extract_pricing(html_pricing)
    pricing.extend(pr)
    print(f"  ✓ {len(pr)} entradas de preço")

# Extract from blog
if html_blog:
    print("\n[3/5] Processando Blog...")
    bl = extract_blog(html_blog)
    news_events.extend(bl.get("news_events", []))
    algorithms.extend(bl.get("algorithms", []))
    clients.extend(bl.get("clients", []))
    print(f"  ✓ {len(bl.get('news_events', []))} notícias")
    print(f"  ✓ {len(bl.get('algorithms', []))} algoritmos")
    print(f"  ✓ {len(bl.get('clients', []))} clientes")

# Extract from docs
if html_docs:
    print("\n[4/5] Processando Docs...")
    dc = extract_docs_braket(html_docs)
    services.extend(dc.get("services", []))
    algorithms.extend(dc.get("algorithms", []))
    print(f"  ✓ {len(dc.get('services', []))} serviços")
    print(f"  ✓ {len(dc.get('algorithms', []))} algoritmos")

# Extract from devices
if html_devices:
    print("\n[5/5] Processando Devices...")
    dv = extract_devices(html_devices)
    devices.extend(dv)
    print(f"  ✓ {len(dv)} dispositivos")

print("\n✓ Processamento completo!")


PROCESSANDO DADOS EXTRAÍDOS

[1/5] Processando Overview...
  ✓ 6 serviços
  ✓ 4 algoritmos
  ✓ 4 clientes

[2/5] Processando Pricing...
  ✓ 22 entradas de preço

[3/5] Processando Blog...
  ✓ 10 notícias
  ✓ 1 algoritmos
  ✓ 2 clientes

[4/5] Processando Docs...
  ✓ 3 serviços
  ✓ 4 algoritmos

[5/5] Processando Devices...
  ✓ 4 dispositivos

✓ Processamento completo!


## 8. Pipeline ETL - Transformação

In [12]:
print("\n" + "=" * 60)
print("TRANSFORMANDO E NORMALIZANDO DADOS")
print("=" * 60)

# Services
print("\n[1/6] Transformando Services...")
df_services = pd.DataFrame(services)
if not df_services.empty:
    df_services["service_name"] = df_services["service_name"].astype(str).str.strip()
    df_services["description"] = df_services["description"].astype(str).str.strip()
    df_services["source_url"] = df_services["source_url"].astype(str)
    df_services["capture_timestamp"] = df_services["capture_timestamp"].astype(str)
    df_services = unique_rows(df_services, ["service_name", "source_url"])
    print(f"  ✓ {len(df_services)} registros únicos")

# Pricing
print("[2/6] Transformando Pricing...")
df_pricing = pd.DataFrame(pricing)
if not df_pricing.empty:
    df_pricing["service_or_device"] = df_pricing["service_or_device"].astype(str).str.strip()
    df_pricing["price_type"] = df_pricing["price_type"].astype(str).str.strip()

    def to_float(v):
        try:
            return float(v)
        except Exception:
            return None

    df_pricing["price_value"] = df_pricing["price_value"].apply(to_float)
    df_pricing["currency"] = df_pricing["currency"].astype(str)
    df_pricing["source_url"] = df_pricing["source_url"].astype(str)
    df_pricing["capture_timestamp"] = df_pricing["capture_timestamp"].astype(str)
    df_pricing = unique_rows(df_pricing, ["service_or_device", "price_type", "source_url"])
    print(f"  ✓ {len(df_pricing)} registros únicos")

# Devices
print("[3/6] Transformando Devices...")
df_devices = pd.DataFrame(devices)
if not df_devices.empty:
    df_devices["device_id"] = df_devices["device_id"].astype(str).str.strip()
    df_devices["provider"] = df_devices["provider"].astype(str).str.strip()
    df_devices["technology"] = df_devices["technology"].astype(str).str.strip()
    df_devices["region_list"] = df_devices["region_list"].astype(str)
    df_devices["source_url"] = df_devices["source_url"].astype(str)
    df_devices["capture_timestamp"] = df_devices["capture_timestamp"].astype(str)
    df_devices = unique_rows(df_devices, ["provider", "technology", "source_url"])
    print(f"  ✓ {len(df_devices)} registros únicos")

# Algorithms
print("[4/6] Transformando Algorithms...")
df_algorithms = pd.DataFrame(algorithms)
if not df_algorithms.empty:
    df_algorithms["algorithm_name"] = df_algorithms["algorithm_name"].astype(str).str.strip()
    df_algorithms["description"] = df_algorithms["description"].astype(str).str.strip()
    df_algorithms["source_url"] = df_algorithms["source_url"].astype(str)
    df_algorithms["capture_timestamp"] = df_algorithms["capture_timestamp"].astype(str)
    df_algorithms = unique_rows(df_algorithms, ["algorithm_name", "source_url"])
    print(f"  ✓ {len(df_algorithms)} registros únicos")

# Clients
print("[5/6] Transformando Clients...")
df_clients = pd.DataFrame(clients)
if not df_clients.empty:
    df_clients["client_name"] = df_clients["client_name"].astype(str).str.strip()
    df_clients["sector"] = df_clients["sector"].astype(str)
    df_clients["use_case_summary"] = df_clients["use_case_summary"].astype(str).str.strip()
    df_clients["source_url"] = df_clients["source_url"].astype(str)
    df_clients["capture_timestamp"] = df_clients["capture_timestamp"].astype(str)
    df_clients = unique_rows(df_clients, ["client_name", "use_case_summary", "source_url"])
    print(f"  ✓ {len(df_clients)} registros únicos")

# News/events
print("[6/6] Transformando News/Events...")
df_news = pd.DataFrame(news_events)
if not df_news.empty:
    df_news["title"] = df_news["title"].astype(str).str.strip()
    df_news["summary"] = df_news["summary"].astype(str)
    df_news["pub_date"] = df_news["pub_date"].apply(lambda d: normalize_date(d) if d else "")
    df_news["source_url"] = df_news["source_url"].astype(str)
    df_news["capture_timestamp"] = df_news["capture_timestamp"].astype(str)
    df_news = unique_rows(df_news, ["title", "source_url"])
    print(f"  ✓ {len(df_news)} registros únicos")

print("\n✓ Transformação completa!")


TRANSFORMANDO E NORMALIZANDO DADOS

[1/6] Transformando Services...
  ✓ 9 registros únicos
[2/6] Transformando Pricing...
  ✓ 22 registros únicos
[3/6] Transformando Devices...
  ✓ 4 registros únicos
[4/6] Transformando Algorithms...
  ✓ 9 registros únicos
[5/6] Transformando Clients...
  ✓ 6 registros únicos
[6/6] Transformando News/Events...
  ✓ 10 registros únicos

✓ Transformação completa!


## 9. Pipeline ETL - Carga (Exportação CSVs)

In [13]:
print("\n" + "=" * 60)
print("EXPORTANDO ARQUIVOS CSV")
print("=" * 60 + "\n")

# Salvar CSVs
df_services.to_csv("services.csv", index=False)
print("✓ services.csv salvo")

df_pricing.to_csv("pricing.csv", index=False)
print("✓ pricing.csv salvo")

df_devices.to_csv("devices.csv", index=False)
print("✓ devices.csv salvo")

df_algorithms.to_csv("algorithms.csv", index=False)
print("✓ algorithms.csv salvo")

df_clients.to_csv("clients.csv", index=False)
print("✓ clients.csv salvo")

df_news.to_csv("news_events.csv", index=False)
print("✓ news_events.csv salvo")

print("\n" + "=" * 60)
print("ETL CONCLUÍDO COM SUCESSO!")
print("=" * 60)


EXPORTANDO ARQUIVOS CSV

✓ services.csv salvo
✓ pricing.csv salvo
✓ devices.csv salvo
✓ algorithms.csv salvo
✓ clients.csv salvo
✓ news_events.csv salvo

ETL CONCLUÍDO COM SUCESSO!


## 10. Visualização dos Resultados

In [14]:
print("\n📊 RESUMO DOS DADOS EXTRAÍDOS\n")

print("=" * 60)
print("SERVICES")
print("=" * 60)
print(df_services.head())
print(f"\nTotal: {len(df_services)} serviços\n")

print("=" * 60)
print("PRICING")
print("=" * 60)
print(df_pricing.head())
print(f"\nTotal: {len(df_pricing)} entradas de preço\n")

print("=" * 60)
print("DEVICES")
print("=" * 60)
print(df_devices.head())
print(f"\nTotal: {len(df_devices)} dispositivos\n")

print("=" * 60)
print("ALGORITHMS")
print("=" * 60)
print(df_algorithms.head())
print(f"\nTotal: {len(df_algorithms)} algoritmos\n")

print("=" * 60)
print("CLIENTS")
print("=" * 60)
print(df_clients.head())
print(f"\nTotal: {len(df_clients)} clientes\n")

print("=" * 60)
print("NEWS & EVENTS")
print("=" * 60)
print(df_news.head())
print(f"\nTotal: {len(df_news)} notícias/eventos\n")


📊 RESUMO DOS DADOS EXTRAÍDOS

SERVICES
                         service_name  \
0                       Amazon Braket   
1                       Braket Direct   
2                   Managed Notebooks   
3  Quantum Simulators (SV1, DM1, TN1)   
4                         Hybrid Jobs   

                                         description  \
0  Serviço de computação quântica gerenciado, ace...   
1  Reserva de acesso dedicado a dispositivos, con...   
2  Ambiente Jupyter gerenciado via SageMaker para...   
3  Simuladores gerenciados de circuitos quânticos...   
4  Execução gerenciada de algoritmos híbridos quâ...   

                       source_url           capture_timestamp  
0  https://aws.amazon.com/braket/  2025-10-12T21:10:13.962170  
1  https://aws.amazon.com/braket/  2025-10-12T21:10:13.962170  
2  https://aws.amazon.com/braket/  2025-10-12T21:10:13.962170  
3  https://aws.amazon.com/braket/  2025-10-12T21:10:13.962170  
4  https://aws.amazon.com/braket/  2025-10-12T21:10:13.9

## 11. Estatísticas Adicionais

In [15]:
print("\n📈 ESTATÍSTICAS DETALHADAS\n")

# Pricing por tipo
print("Distribuição de Preços por Tipo:")
print(df_pricing.groupby('price_type')['price_value'].describe())
print()

# Dispositivos por provedor
print("\nDispositivos por Provedor:")
print(df_devices['provider'].value_counts())
print()

# Algoritmos mais mencionados
print("\nAlgoritmos Únicos:")
print(df_algorithms['algorithm_name'].value_counts())


📈 ESTATÍSTICAS DETALHADAS

Distribuição de Preços por Tipo:
                        count         mean          std        min  \
price_type                                                           
free_minutes_per_month    1.0    60.000000          NaN    60.0000   
per_minute                1.0     0.075000          NaN     0.0750   
per_shot                  6.0     0.020658     0.031128     0.0009   
per_task                  6.0     0.300000     0.000000     0.3000   
reservation_hour          6.0  4875.000000  1985.887711  2500.0000   

                                25%        50%       75%       max  
price_type                                                          
free_minutes_per_month    60.000000    60.0000    60.000    60.000  
per_minute                 0.075000     0.0750     0.075     0.075  
per_shot                   0.001487     0.0058     0.025     0.080  
per_task                   0.300000     0.3000     0.300     0.300  
reservation_hour        3250.00000

## 12. Download dos Arquivos (Google Colab)

Execute esta célula para fazer download dos CSVs gerados:

In [17]:
# Descomente as linhas abaixo para fazer download dos arquivos no Google Colab
from google.colab import files
#files.download('services.csv')
#files.download('pricing.csv')
#files.download('devices.csv')
#files.download('algorithms.csv')
#files.download('clients.csv')
#files.download('news_events.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>